# Lab 10: Inner Products, Projections, QR, and Adjoints — Independent Study

This notebook includes worked solutions and similar practice questions for Chapter 10.

In [ ]:
import numpy as np
from numpy.linalg import norm, qr, solve, lstsq
np.set_printoptions(precision=4, suppress=True)

## 1. Inner product, norms, and angle

For $u=(1,2,-1)$ and $v=(3,0,4)$, compute $u\cdot v$, norms, and the angle.

In [ ]:
u = np.array([1,2,-1.])
v = np.array([3,0,4.])
dot = u @ v
angle = np.arccos(dot/(norm(u)*norm(v)))
dot, norm(u), norm(v), angle, angle*180/np.pi

**Answer.** The dot product is $-1$, the norms are $\sqrt6$ and $5$, and $\cos\theta=-1/(5\sqrt6)$.

**Similar practice.** Repeat for $a=(2,-1,2)$ and $b=(1,3,0)$.

In [ ]:
a = np.array([2,-1,2.])
b = np.array([1,3,0.])
dot = a @ b
angle = np.arccos(dot/(norm(a)*norm(b)))
dot, norm(a), norm(b), angle

## 2. Projection onto a line

Project $y=(4,1,2)$ onto $\operatorname{span}\{w\}$, where $w=(1,2,0)$.

In [ ]:
y = np.array([4,1,2.])
w = np.array([1,2,0.])
proj = (y @ w)/(w @ w)*w
r = y - proj
proj, r, r @ w

**Answer.** The projection is $(6/5,12/5,0)$ and the residual is $(14/5,-7/5,2)$. The residual is orthogonal to $w$.

**Similar practice.** Project $y=(2,3,1)$ onto $\operatorname{span}\{(1,1,0)\}$.

In [ ]:
y = np.array([2,3,1.])
w = np.array([1,1,0.])
proj = (y @ w)/(w @ w)*w
r = y - proj
proj, r, r @ w

## 3. Gram--Schmidt

Apply Gram--Schmidt to $b_1=(1,1,1)$ and $b_2=(1,0,1)$.

In [ ]:
def classical_gram_schmidt(A):
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m,n))
    R = np.zeros((n,n))
    for j in range(n):
        v = A[:,j].copy()
        for i in range(j):
            R[i,j] = Q[:,i] @ A[:,j]
            v = v - R[i,j]*Q[:,i]
        R[j,j] = norm(v)
        Q[:,j] = v/R[j,j]
    return Q, R

A = np.array([[1,1],[1,0],[1,1.]], dtype=float)
Q, R = classical_gram_schmidt(A)
Q, R, Q.T @ Q, Q @ R

**Answer.** An orthonormal basis is $(1,1,1)/\sqrt3$ and $(1,-2,1)/\sqrt6$.

**Similar practice.** Apply Gram--Schmidt to $(1,1,0)$ and $(1,0,1)$.

In [ ]:
B = np.array([[1,1],[1,0],[0,1.]], dtype=float)
Q, R = classical_gram_schmidt(B)
Q, R, Q.T @ Q

## 4. QR factorization

For a matrix with independent columns, QR packages Gram--Schmidt as $A=QR$.

In [ ]:
A = np.array([[1,1],[1,0],[1,1.]], dtype=float)
Q_np, R_np = np.linalg.qr(A)
Q_np, R_np, Q_np.T @ Q_np, Q_np @ R_np

Python may choose signs differently from a hand solution. Both are valid if $Q^TQ=I$ and $QR=A$.

## 5. Least squares by QR

Fit $bpprox c_0+c_1t$ using $t=1,2,3,4$.

In [ ]:
t = np.array([1,2,3,4.])
A = np.column_stack([np.ones_like(t), t])
b = np.array([1.1, 1.9, 3.2, 3.9])
Q, R = np.linalg.qr(A)
x_qr = solve(R, Q.T @ b)
x_lstsq, *_ = lstsq(A, b, rcond=None)
x_qr, x_lstsq

**Answer.** The coefficients give the least-squares line $\hat b=c_0+c_1t$.

**Similar practice.** Replace $b$ by $(2,2.9,4.1,5.2)$.

In [ ]:
b = np.array([2,2.9,4.1,5.2])
Q, R = np.linalg.qr(A)
solve(R, Q.T @ b)

## 6. Orthogonal matrices

Check whether $U^TU=I$.

In [ ]:
U = np.array([[1,1],[1,-1.]])/np.sqrt(2)
U.T @ U, np.linalg.inv(U), U.T

**Answer.** This matrix is orthogonal, so $U^{-1}=U^T$. Since it is symmetric, $U^{-1}=U$.

## 7. Complex adjoints

Compute $A^*=\overline A^T$.

In [ ]:
A = np.array([[1,1j],[2,1-1j]], dtype=complex)
A.conj().T

**Answer.** $A^*=\begin{bmatrix}1&2\\-i&1+i\end{bmatrix}$.

**Similar practice.** Compute the adjoint of $B=\begin{bmatrix}2&3i\\1-i&4\end{bmatrix}$.

In [ ]:
B = np.array([[2,3j],[1-1j,4]], dtype=complex)
B.conj().T

## 8. Challenge: normal equations versus QR

Compare least squares using normal equations and QR for an ill-conditioned Vandermonde matrix.

In [ ]:
t = np.linspace(0, 1, 12)
A = np.vander(t, 8, increasing=True)
true_x = np.ones(8)
b = A @ true_x + 1e-8*np.random.default_rng(0).normal(size=12)
# Normal equations
x_ne = solve(A.T @ A, A.T @ b)
# QR
Q, R = qr(A, mode='reduced')
x_qr = solve(R, Q.T @ b)
print('condition(A)=', np.linalg.cond(A))
print('condition(A^T A)=', np.linalg.cond(A.T @ A))
print('normal equations error=', norm(x_ne-true_x))
print('QR error=', norm(x_qr-true_x))

The condition number of $A^TA$ is approximately the square of the condition number of $A$, so QR is usually more stable.